In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

import sys

scripts_path = "/local/scratch/lslagter/MSC_project/STRW-MSC/scripts"
scripts_path = "/home/slagter/Astronomy_master/STRW-MSC/SF_in_NGC/scripts"
from pathlib import Path

scripts_path = Path.cwd().parents[1]
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

import Config
from Config import (
    CATALOGUE_JEROEN,
    POSITIONS_F115W,
    GAIA_CATALOGUE,
    MPLSTYLE,
    SB_PARAM_DIR,
    PHOT_DIR,
    PLOTS_DIR,
    LEVEL2_DIR,
    LEVEL3_DIR,
    STARBUG2_BIN,
)
import phot_toolkit as PTK
import photometric_pipeline as PPL
import plot_toolkit as PLOT

import numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from astropy.stats import knuth_bin_width
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import astropy.units as u

if MPLSTYLE.exists():
    plt.style.use(str(MPLSTYLE))

working_directory = f"{LEVEL2_DIR}/mastDownload/JWST"
PHOTOMETRIC_pipeline = PPL.Pipeline_level_2_data(working_directory=working_directory)

import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack

In [ ]:
def merge_catalogues(deep_file, shallow_file, out_file, filt="F115W",
                     match_arcsec=0.06, max_err=2.0):
    def load(f):
        t = Table.read(f, hdu=1)
        return t[(t[f"e{filt}"] <= max_err) & (t["flag"] == 0)]

    deep, shallow = load(deep_file), load(shallow_file)

    c_deep = SkyCoord(deep["RA"], deep["DEC"], unit="deg")
    c_shal = SkyCoord(shallow["RA"], shallow["DEC"], unit="deg")

    _, sep, _ = c_shal.match_to_catalog_sky(c_deep)
    keep = sep > match_arcsec * u.arcsec

    merged = vstack([deep, shallow[keep]], join_type="outer")
    merged.write(out_file, overwrite=True)

    print(f"deep: {len(deep)}, shallow: {len(shallow)}, "
          f"duplicates dropped: {(~keep).sum()}, merged: {len(merged)}")
    return merged

In [ ]:
filt        = 'F115W'
path_short =  f'{LEVEL3_DIR}/mosaic_{filt}_short_exposure-ap.fits'
path_long   =  f'{LEVEL3_DIR}/mosaic_{filt}_long_exposure-ap.fits'

merge_catalogues(
   path_short,
   path_long,
    f"{LEVEL3_DIR}/F115W_merged_ap.fits",
    filt = filt,
    match_arcsec=0.06, max_err=2.0
)

deep: 1979389, shallow: 971671, duplicates dropped: 2640, merged: 2948420


Catalogue_Number,xcentroid,ycentroid,RA,DEC,sharpness,roundness1,roundness2,smoothness,flux,eflux,sky,flag,F115W,eF115W
bytes9,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,uint16,float64,float64
CN0,9995.041262723702,7.637353576839299,15.231483172527145,-72.20665957148282,0.8711443063616999,-0.3470957279205322,-0.8265597691818666,0.7326846087698712,1.127114684546277e-06,2.225528314444851e-08,3.833406925224381e-09,0,23.770079730001008,0.021229342766485103
CN2,10009.803732504239,9.17300184836532,15.231407273965532,-72.20678620897822,0.841357863811993,-0.26002565026283264,0.2428148971669669,0.5393999291270875,6.00033921182397e-08,1.2645167273671938e-08,8.75953726220402e-10,0,26.954560493349568,0.2076280101911534
CN3,10001.677633443385,11.078247147154837,15.231371210862246,-72.20671465356763,0.8329953216683788,0.205405592918396,-1.0510685763404923,0.7968020977652367,1.5967274890017382e-08,1.289770916367866e-08,2.0057571159526333e-09,0,28.39192299475325,0.6428512995309636
CN4,10017.156522569909,10.352963717114449,15.231357722999894,-72.20684900496978,0.9650900972554918,-0.13365153968334198,-0.3644610371703529,0.8414208943011742,1.7361603240071494e-08,1.2727118585487259e-08,1.8840969895350668e-09,0,28.301025432033803,0.5970347800898003
CN5,9936.836906406435,11.20360433969368,15.231510141444296,-72.20615381439731,0.7966150560779862,0.2096884399652481,0.7684182375605884,0.8503915667204909,1.0634983377471147e-08,8.52538719051791e-09,1.4915819690486387e-09,0,28.83315796141258,0.6391677219740783
CN6,9981.289081176274,13.472446075707516,15.231348256272428,-72.2065367217938,0.9100208757169871,0.0036735571920871735,0.8798630751780974,0.919504677027831,4.854145213815091e-08,1.420092998956171e-08,4.557867860910392e-09,0,27.184718089264813,0.2786205978851524
CN10,9944.351509262631,14.404678763988551,15.231403039979291,-72.20621665265247,0.7439009432838566,-1.1427994966506958,-0.15132243760471178,0.593016747693623,1.5877294535705245e-08,1.359519373398628e-08,2.1426558305392973e-09,0,28.39805874663547,0.6716007554304251
CN14,9915.031816689356,16.68296073750804,15.231402990729828,-72.20596156051631,0.6804922811211934,0.13180933892726898,-0.044008893539609975,0.8572596749586274,1.1243403745602967e-08,1.3844068034172093e-08,4.120952024067037e-09,0,28.77275548433807,0.8713977267620399
CN15,10000.521800620676,15.825191073482687,15.231239409813975,-72.20670147011116,0.5084433008462191,-0.9873952269554138,-0.20457848583932295,0.7864089712253978,1.8473869626267453e-08,1.3519999938677471e-08,2.28299401605625e-09,0,28.233605314009864,0.5962722387802473
